# 35. The Heliospheric Magnetic Field - Implementation / 태양권 자기장 구현

**Paper**: Owens, M. J. and Forsyth, R. J., *Living Rev. Solar Phys.* **10**, 5 (2013). DOI: 10.12942/lrsp-2013-5

This notebook reproduces core analytical results of Owens and Forsyth (2013):
1. Parker spiral components B_R(R,theta) and B_phi(R,theta)
2. Streamlines of the HMF in the ecliptic plane
3. Parker angle psi(R) for solar wind speeds 400, 700, 1000 km/s
4. Ulysses-like latitude invariance of R^2 |B_R|
5. Warped heliospheric current sheet (HCS) in 3D (tilt angle alpha)
6. Sector-crossing signatures at Earth over one solar rotation
7. Reference HMF magnitudes at 1, 5, 10, 30 AU

이 노트북은 Owens와 Forsyth(2013)의 핵심 해석적 결과를 재현한다: Parker 나선, 태양풍 속도별 Parker 각, Ulysses 위도 불변성, 3D HCS, 섹터 교차, 기준 거리 HMF 크기.

## 0. Imports and Constants / 불러오기 및 상수

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

# Physical constants (SI units).
AU = 1.495978707e11
R_SUN = 6.957e8
OMEGA_SUN = 2.66e-6
R0 = 2.5 * R_SUN
BR0 = 1800e-9
VSW_DEFAULT = 400e3

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
print(f"AU = {AU:.3e} m")
print(f"Omega_Sun = {OMEGA_SUN:.3e} rad/s")
print(f"R0 = 2.5 R_Sun = {R0:.3e} m")

## 1. Parker Spiral Components / Parker 나선 성분

From Eqs. (1)-(3) of the paper:

$$B_R(R,\theta) = B_R(R_0)(R_0/R)^2,\quad B_\phi = -B_R(R_0) \Omega R_0^2 \sin\theta / (V_R R),\quad \tan\psi = \Omega R \sin\theta / V_R$$

논문의 식 (1)-(3)으로부터 방사·방위각 성분과 Parker 각을 계산한다.

In [ ]:
def parker_BR(R, BR0=BR0, R0=R0):
    """Radial component of Parker spiral field.

    Args:
        R: Heliocentric distance [m].
        BR0: Radial field at source surface [T].
        R0: Source surface radius [m].

    Returns:
        Radial field component B_R [T].
    """
    return BR0 * (R0 / R) ** 2


def parker_Bphi(R, theta, vsw=VSW_DEFAULT, BR0=BR0, R0=R0, omega=OMEGA_SUN):
    """Azimuthal component of Parker spiral field.

    Args:
        R: Heliocentric distance [m].
        theta: Colatitude [rad] (pi/2 = ecliptic).
        vsw: Solar wind speed [m/s].
        BR0: Radial field at source surface [T].
        R0: Source surface radius [m].
        omega: Solar rotation rate [rad/s].

    Returns:
        Azimuthal field component B_phi [T].
    """
    return -BR0 * omega * R0 ** 2 * np.sin(theta) / (vsw * R)


def parker_angle(R, theta=np.pi / 2, vsw=VSW_DEFAULT, omega=OMEGA_SUN):
    """Parker angle psi between B and the radial direction.

    Args:
        R: Heliocentric distance [m].
        theta: Colatitude [rad]; default = ecliptic.
        vsw: Solar wind speed [m/s].
        omega: Solar rotation rate [rad/s].

    Returns:
        Parker angle psi [rad].
    """
    return np.arctan2(omega * R * np.sin(theta), vsw)


def parker_magnitude(R, theta, vsw=VSW_DEFAULT, BR0=BR0, R0=R0, omega=OMEGA_SUN):
    """Total Parker spiral field magnitude.

    Args:
        R: Heliocentric distance [m].
        theta: Colatitude [rad].
        vsw: Solar wind speed [m/s].
        BR0: Radial field at source surface [T].
        R0: Source surface radius [m].
        omega: Solar rotation rate [rad/s].

    Returns:
        |B| [T].
    """
    br = parker_BR(R, BR0=BR0, R0=R0)
    bp = parker_Bphi(R, theta, vsw=vsw, BR0=BR0, R0=R0, omega=omega)
    return np.sqrt(br ** 2 + bp ** 2)


# Sanity check: Parker angle at 1 AU, 400 km/s, equator should be ~45 deg.
psi_1au = np.degrees(parker_angle(1.0 * AU))
print(f"Parker angle at 1 AU (400 km/s, equator) = {psi_1au:.2f} deg (expected ~45 deg)")

## 2. Streamlines of the Parker Spiral in the Ecliptic / 황도면의 Parker 나선 유선

In the ecliptic, a field line from footpoint phi_0 satisfies phi(R) = phi_0 - Omega*(R - R_0)/V_R.

황도면에서 발판 phi_0으로부터의 자기력선은 위 식을 따른다.

In [ ]:
def parker_fieldline(phi0, R, vsw=VSW_DEFAULT, R0=R0, omega=OMEGA_SUN):
    """Longitude of a Parker-spiral field line vs. R.

    Args:
        phi0: Footpoint longitude at R0 [rad].
        R: Heliocentric distance [m] (array).
        vsw: Solar wind speed [m/s].
        R0: Source surface radius [m].
        omega: Solar rotation rate [rad/s].

    Returns:
        Longitude phi(R) [rad].
    """
    return phi0 - omega * (R - R0) / vsw


R = np.linspace(R0, 2.0 * AU, 400)
n_lines = 12
phi0s = np.linspace(0, 2 * np.pi, n_lines, endpoint=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 6), subplot_kw={"projection": "polar"})
for ax, vsw, label in zip(axes, [400e3, 800e3], ["V = 400 km/s", "V = 800 km/s"]):
    for phi0 in phi0s:
        phi = parker_fieldline(phi0, R, vsw=vsw)
        color = "tab:red" if (phi0 < np.pi) else "tab:blue"
        ax.plot(phi, R / AU, color=color, lw=1.2)
    theta_c = np.linspace(0, 2 * np.pi, 400)
    ax.plot(theta_c, np.ones_like(theta_c), "k--", lw=0.8, label="1 AU")
    ax.set_ylim(0, 2)
    ax.set_title(f"Parker spiral ({label})")
    ax.set_rlabel_position(135)
    ax.grid(True, alpha=0.3)

plt.suptitle("Ecliptic-plane Parker spiral streamlines / 황도면의 Parker 나선 유선", y=1.02)
plt.tight_layout()
plt.show()

**Observation / 관측**: Faster solar wind (right) produces a less tightly wound spiral, consistent with tan(psi) = Omega*R/V_R. Red/blue sketch alternating sectors separated by the HCS.

더 빠른 태양풍(오른쪽)은 덜 감긴 나선을 만든다. 빨강/파랑은 HCS로 분리된 교대 섹터를 묘사한다.

## 3. Parker Angle vs. Solar Wind Speed / 태양풍 속도별 Parker 각

Compute psi(R) for V_R in {400, 700, 1000} km/s from 0.1 to 5 AU.

In [ ]:
R_arr = np.linspace(0.1 * AU, 5.0 * AU, 300)
speeds_kms = [400, 700, 1000]

fig, ax = plt.subplots(figsize=(7, 5))
for v_kms in speeds_kms:
    psi_deg = np.degrees(parker_angle(R_arr, vsw=v_kms * 1e3))
    ax.plot(R_arr / AU, psi_deg, lw=2, label=f"V = {v_kms} km/s")

ax.axvline(1, color="k", ls=":", lw=1)
ax.axhline(45, color="grey", ls=":", lw=1)
ax.set_xlabel("Heliocentric distance [AU] / 태양중심 거리")
ax.set_ylabel("Parker angle psi [deg] / Parker 각")
ax.set_title("Parker angle vs distance / 거리에 따른 Parker 각")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Parker angle at 1 AU / 1 AU에서의 Parker 각:")
for v_kms in speeds_kms:
    psi_1 = np.degrees(parker_angle(1.0 * AU, vsw=v_kms * 1e3))
    print(f"  V = {v_kms:4d} km/s  ->  psi = {psi_1:5.2f} deg")

## 4. Ulysses Latitude Invariance: R^2 |B_R| / Ulysses 위도 불변성

Smith and Balogh (1995, 2003) showed R^2 |B_R| is latitude-independent. Verify this in the Parker model and compare |B| at different latitudes.

Smith & Balogh(1995, 2003)는 R^2 |B_R|이 위도에 무관함을 보였다. Parker 모델에서 이를 확인한다.

In [ ]:
R_arr = np.linspace(0.3 * AU, 10.0 * AU, 300)
latitudes_deg = [0, 30, 60, 80]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for lat in latitudes_deg:
    theta = np.pi / 2 - np.radians(lat)
    Bmag = parker_magnitude(R_arr, theta)
    BR_only = np.abs(parker_BR(R_arr))
    axes[0].loglog(R_arr / AU, Bmag * 1e9, lw=2, label=f"lat = {lat} deg")
    axes[1].plot(R_arr / AU, (R_arr ** 2) * BR_only / AU ** 2 * 1e9, lw=2, label=f"lat = {lat} deg")

axes[0].set_xlabel("R [AU]")
axes[0].set_ylabel("|B| [nT]")
axes[0].set_title("|B|(R) at different latitudes / 위도별 |B|(R)")
axes[0].grid(True, which="both", alpha=0.3)
axes[0].legend()

axes[1].set_xlabel("R [AU]")
axes[1].set_ylabel("R^2 |B_R| / AU^2 [nT]")
axes[1].set_title("Ulysses result: R^2 |B_R| vs R / Ulysses 결과")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

**Observation / 관측**: All curves on the right panel coincide - R^2 |B_R| is latitude-invariant. A single near-Earth measurement fixes Phi_OSF = 4*pi*R^2 |B_R|.

오른쪽 그림의 모든 곡선이 일치한다. 단일점 측정으로 총 열린 태양 플럭스가 결정된다.

## 5. Warped Heliospheric Current Sheet (HCS) in 3D / 3D 뒤틀린 HCS

For a tilted magnetic dipole (tilt angle alpha) the HCS surface is

theta_HCS(phi, R) = pi/2 + alpha * sin(phi - Omega*(R - R0)/V_R)

기울어진 쌍극자에 대해 HCS는 위와 같은 파동형 표면이다.

In [ ]:
def hcs_surface(R_grid, phi_grid, alpha_rad, vsw=VSW_DEFAULT, omega=OMEGA_SUN, R0=R0):
    """Cartesian coordinates (x, y, z) of the warped HCS over an (R, phi) grid.

    Args:
        R_grid: Radial distances [m] (2D array).
        phi_grid: Azimuthal angles [rad] (2D array).
        alpha_rad: Dipole tilt angle [rad].
        vsw: Solar wind speed [m/s].
        omega: Solar rotation rate [rad/s].
        R0: Source surface radius [m].

    Returns:
        Tuple (x, y, z) of HCS surface coordinates [m].
    """
    phi_effective = phi_grid - (omega / vsw) * (R_grid - R0)
    theta = np.pi / 2 + alpha_rad * np.sin(phi_effective)
    x = R_grid * np.sin(theta) * np.cos(phi_grid)
    y = R_grid * np.sin(theta) * np.sin(phi_grid)
    z = R_grid * np.cos(theta)
    return x, y, z


R_vals = np.linspace(0.2 * AU, 2.0 * AU, 80)
phi_vals = np.linspace(0, 2 * np.pi, 120)
Rg, Pg = np.meshgrid(R_vals, phi_vals)

alpha_deg = 20.0
x, y, z = hcs_surface(Rg, Pg, np.radians(alpha_deg))

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

surf = ax.plot_surface(x / AU, y / AU, z / AU, cmap="coolwarm", linewidth=0, antialiased=True, alpha=0.85)

t_c = np.linspace(0, 2 * np.pi, 200)
ax.plot(np.cos(t_c), np.sin(t_c), np.zeros_like(t_c), "k--", lw=1.5, label="Earth orbit")
ax.scatter([0], [0], [0], color="gold", s=120, label="Sun")

ax.set_xlabel("X [AU]")
ax.set_ylabel("Y [AU]")
ax.set_zlabel("Z [AU]")
ax.set_title(f"Warped HCS (tilt alpha = {alpha_deg:.0f} deg) / 뒤틀린 HCS")
ax.legend()
fig.colorbar(surf, shrink=0.5, aspect=12, label="z [AU]")
plt.tight_layout()
plt.show()

**Observation / 관측**: HCS forms a wavy ballerina-skirt shape around the Sun. Earth orbit (dashed at z=0) crosses the HCS multiple times per rotation, producing the observed sector structure.

HCS는 태양 주위의 물결치는 발레리나 치마 형태이다. 지구 궤도는 한 자전당 HCS를 여러 번 교차한다.

## 6. Sector Crossings at Earth over One Solar Rotation / 한 자전 동안의 섹터 교차

Earth is fixed at (R=1 AU, z=0); HCS rotates beneath. B_R polarity switches at each crossing. Top = pure dipole (2 sectors); bottom = dipole+quadrupole (4 sectors).

지구는 1 AU, z=0에 고정되고 HCS는 그 아래에서 회전한다. 교차 시마다 B_R 부호가 바뀐다.

In [ ]:
def br_sign_at_earth(days, n_harmonic=1, alpha_deg=20.0):
    """Signed radial-field proxy at Earth as the tilted-dipole HCS rotates.

    Args:
        days: Time array [day].
        n_harmonic: 1 for pure dipole (2-sector), 2 for dipole+quadrupole (4-sector).
        alpha_deg: Dipole tilt [deg] (sets amplitude).

    Returns:
        Array of sign-valued polarity (+1 outward / -1 inward).
    """
    omega_syn = 2 * np.pi / (27.27 * 86400)
    t_sec = days * 86400
    phase = omega_syn * t_sec
    amplitude = np.tanh(5 * np.radians(alpha_deg))
    return -np.sign(np.sin(n_harmonic * phase)) * amplitude


days = np.linspace(0, 27.3, 1500)

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

for ax, n_harm, label_en, label_kr in [
    (axes[0], 1, "Pure dipole (2-sector)", "쌍극자 (2-섹터)"),
    (axes[1], 2, "Dipole+quadrupole (4-sector)", "쌍극자+사중극 (4-섹터)"),
]:
    br = br_sign_at_earth(days, n_harmonic=n_harm, alpha_deg=20.0)
    ax.fill_between(days, br, 0, where=(br > 0), color="tab:red", alpha=0.5, label="outward (+B_R)")
    ax.fill_between(days, br, 0, where=(br < 0), color="tab:blue", alpha=0.5, label="inward (-B_R)")
    ax.axhline(0, color="k", lw=0.6)
    ax.set_ylabel("sign(B_R)")
    ax.set_title(f"{label_en} / {label_kr}")
    ax.set_ylim(-1.2, 1.2)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right", fontsize=8)

axes[1].set_xlabel("Time [day] / 시간")
plt.suptitle("Sector crossings at Earth over one solar rotation / 한 자전 동안의 섹터 교차", y=1.02)
plt.tight_layout()
plt.show()

## 7. HMF Magnitude at Reference Distances / 기준 거리의 HMF 크기

Tabulate B_R, |B_phi|, |B|, and psi at 1, 5, 10, 30 AU for a 400 km/s equatorial wind, with B_R(R_0) calibrated so that |B|(1 AU) = 5 nT.

적도 400 km/s 태양풍에 대해 1, 5, 10, 30 AU에서의 자기장을 표로 정리한다.

In [ ]:
R_ref_AU = np.array([1.0, 5.0, 10.0, 30.0])
theta_eq = np.pi / 2

# Calibrate BR0 so |B|(1 AU) = 5 nT; |B| = BR0 (R0/AU)^2 * sqrt(2) at psi=45 deg.
target_B_1AU = 5e-9
BR0_cal = target_B_1AU / ((R0 / AU) ** 2 * np.sqrt(2))
print(f"Calibrated BR0 = {BR0_cal*1e9:.3e} nT (gives |B|(1 AU) ~ 5 nT)")
print()

header = "{:>8} | {:>10} | {:>14} | {:>10} | {:>10}".format("R [AU]", "B_R [nT]", "|B_phi| [nT]", "|B| [nT]", "psi [deg]")
print(header)
print("-" * 62)
for R_au in R_ref_AU:
    R_m = R_au * AU
    br = parker_BR(R_m, BR0=BR0_cal)
    bp = np.abs(parker_Bphi(R_m, theta_eq, BR0=BR0_cal))
    bmag = parker_magnitude(R_m, theta_eq, BR0=BR0_cal)
    psi = np.degrees(parker_angle(R_m))
    print(f"{R_au:>8.1f} | {br*1e9:>10.3e} | {bp*1e9:>14.3e} | {bmag*1e9:>10.3e} | {psi:>10.2f}")

**Interpretation / 해석**:
- 1 AU: |B| ~ 5 nT, psi ~ 45 deg (radial and azimuthal components equal).
- 5 AU: |B_phi| dominates; psi ~ 80 deg.
- 30 AU (Voyager range): psi ~ 89 deg; |B| ~ 1/R.

1 AU에서 방사·방위각 성분이 동등; 5 AU에서 방위각 성분 지배; 30 AU에서 |B| ~ 1/R.

## 8. A>0 vs A<0 Polarity Patterns: 22-year GCR Modulation / 22년 GCR 조절

Following Jokipii et al. (1977), positive GCRs drift differently in A>0 vs A<0 epochs, producing alternating peak and dome cosmic-ray modulation profiles.

Jokipii et al.(1977)에 따라 A>0, A<0 시대에 GCR 드리프트가 달라 피크와 돔 프로파일이 교대된다.

In [ ]:
time = np.linspace(0, 22, 500)  # Two 11-year cycles.
# A>0: peaked minima (sharp GCR max during solar min).
gcr_A_pos = 1 - 0.4 * np.sin(np.pi * time / 11) ** 2
# A<0: dome-like (flatter near max).
gcr_A_neg = 1 - 0.4 * np.abs(np.sin(np.pi * time / 11)) ** 0.5

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(time, gcr_A_pos, lw=2, color="tab:purple", label="A>0 (peaked)")
ax.plot(time, gcr_A_neg, lw=2, color="tab:green", label="A<0 (dome-like)")
ax.set_xlabel("Time [year] / 시간")
ax.set_ylabel("GCR intensity (normalized) / 우주선 강도")
ax.set_title("22-year GCR modulation: A>0 vs A<0 / 22년 GCR 조절")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Summary / 요약

This notebook has reproduced the core quantitative pillars of Owens and Forsyth (2013):

1. **Parker spiral field** - analytical expressions and ecliptic visualization.
2. **Parker angle varies with V_R** - ~45 deg at 400 km/s, ~30 deg at 700 km/s, ~22 deg at 1000 km/s at Earth.
3. **Ulysses latitude invariance** - R^2 |B_R| flat with latitude, enabling single-point OSF estimation.
4. **Warped HCS** - 3D ballerina-skirt surface cut by Earth orbit multiple times per rotation.
5. **Sector structure** - 2-sector (dipole) vs 4-sector (dipole+quadrupole).
6. **Reference values** - |B|(1 AU) ~ 5 nT, |B|(5 AU) ~ 0.8 nT, |B|(30 AU) ~ 0.13 nT.
7. **22-year GCR modulation** - A>0 peak vs A<0 dome profiles.

이 노트북은 Owens & Forsyth(2013)의 핵심 정량적 기둥을 재현했다: Parker 나선, 속도별 Parker 각, Ulysses 위도 불변성, 뒤틀린 HCS 3D 시각화, 섹터 구조, 기준 거리 크기, 22년 GCR 조절.